# AD 599 Term Project: Starter Notebook

This notebook gets your team's data ready. It does five things, top to bottom:

1. Downloads your team's files from the course site
2. Checks every file against its published checksum
3. Reads the files into pandas tables
4. Saves those tables, unchanged, as `raw_` tables in your SQLite database (`team.db`)
5. Gives you a first look with SQL

**How to use it**

- Change `TEAM` in the settings cell below, then run **Kernel > Restart & Run All**.
- Run it with `BATCH = "A"` now. Batch B is released in class on **Monday, November 30**: change `BATCH` to `"B"` and run it again. It adds batch B next to batch A and never duplicates rows.
- Running it twice is safe: files you already have are not downloaded again, and batches already in the database are skipped.
- Never edit anything in `data/raw/`. Every change to the data happens in your own notebooks, in Python or SQL, where it can be rerun and checked.

## 0. Settings (the only cell you change)

In [ ]:
TEAM = 0          # your team number, 1 to 7
BATCH = "A"       # "A" now; "B" from November 30

BASE_URL = "https://met-ad-599.github.io/data/term-project/"
DATA_DIR = "data/raw"
DB_PATH = "team.db"

In [ ]:
import hashlib
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path(DATA_DIR)
DB_PATH = Path(DB_PATH)
assert TEAM in range(1, 8), "Set TEAM to your team number (1 to 7) in the settings cell, then run all cells again"
assert BATCH in ("A", "B"), "BATCH must be \"A\" or \"B\""
print("Python packages ready. pandas", pd.__version__)

## 1. Download your team's files

The course site publishes, for every team and batch, a `manifest.json`: the list of files with their size and SHA-256 checksum. A checksum is a fingerprint of a file: if even one byte changes, the checksum changes. The cell below downloads each file you do not already have and stops if any checksum does not match.

In [ ]:
def get_json(url):
    """Download a JSON document and return it as Python objects."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def sha256_of(path):
    """Return the SHA-256 checksum of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


index = get_json(BASE_URL + "index.json")
team = index["teams"][str(TEAM)]
print(f"Team {TEAM}: {team['name']}")
print(f"Source: {team['source']} | {team['entities']}")

team_url = BASE_URL + f"team_{TEAM:02d}/batch_{BATCH}/"
response = requests.get(team_url + "manifest.json", timeout=60)
if response.status_code == 404:
    raise RuntimeError(f"Batch {BATCH} is not released yet. Batch B is released in class on November 30.")
response.raise_for_status()
manifest = response.json()

batch_dir = DATA_DIR / f"batch_{BATCH}"
batch_dir.mkdir(parents=True, exist_ok=True)
(batch_dir / "manifest.json").write_text(json.dumps(manifest, indent=1))

log_rows = []
for f in manifest["files"]:
    target = batch_dir / f["path"]
    if target.exists() and sha256_of(target) == f["sha256"]:
        status = "already had it"
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        download = requests.get(team_url + f["path"], timeout=300)
        download.raise_for_status()
        target.write_bytes(download.content)
        status = "downloaded"
    assert sha256_of(target) == f["sha256"], f"Checksum does not match for {f['path']}: delete that file and run this cell again"
    log_rows.append({
        "batch": BATCH,
        "path": f["path"],
        "url": team_url + f["path"],
        "bytes": f["bytes"],
        "sha256": f["sha256"],
        "status": status,
        "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

download_log = pd.DataFrame(log_rows)
download_log.to_csv(batch_dir / "download_log.csv", index=False)
print(download_log["status"].value_counts().to_string())
print(f"{len(download_log)} files in {batch_dir}, every checksum matches")

## 2. Read the files into pandas

The raw files are nested JSON (SEC, FDIC) or CSV (FRED). These functions turn them into flat tables, **without changing any value**:

- **SEC:** every reported number becomes one row in `raw_sec_facts`; concept labels go to `raw_sec_concept`; the company profile and filing list (batch A only) go to `raw_sec_company` and `raw_sec_filing`.
- **FDIC:** the bank profiles (batch A only) go to `raw_fdic_institution`; every bank-quarter record goes to `raw_fdic_financials`.
- **FRED (Team 7):** every observation goes to `raw_fred_observation`, with values kept as text exactly as published; the series list goes to `raw_fred_series`.

Every row keeps `source_file`, so you can always trace it back to the file it came from.

In [ ]:
def to_text_if_nested(df):
    """SQLite cannot store lists or dicts: keep them as JSON text instead."""
    for col in df.columns:
        if df[col].map(lambda v: isinstance(v, (list, dict))).any():
            df[col] = df[col].map(lambda v: json.dumps(v) if isinstance(v, (list, dict)) else v)
    return df


def read_sec(batch_dir):
    facts, concepts, companies, filings = [], [], [], []
    for path in sorted(batch_dir.glob("*_companyfacts.json")):
        ticker = path.name.split("_")[0]
        data = json.loads(path.read_text())
        for taxonomy, concept_map in data["facts"].items():
            for concept, detail in concept_map.items():
                concepts.append({"ticker": ticker, "taxonomy": taxonomy, "concept": concept,
                                 "label": detail.get("label"), "description": detail.get("description"),
                                 "source_file": path.name})
                for unit, observations in detail["units"].items():
                    for obs in observations:
                        facts.append({
                            "ticker": ticker, "cik": data["cik"], "taxonomy": taxonomy, "concept": concept,
                            "unit": unit, "start": obs.get("start"), "end": obs["end"], "val": obs["val"],
                            "accn": obs["accn"], "fy": obs.get("fy"), "fp": obs.get("fp"), "form": obs["form"],
                            "filed": obs["filed"], "frame": obs.get("frame"), "source_file": path.name,
                        })
    for path in sorted(batch_dir.glob("*_submissions.json")):
        ticker = path.name.split("_")[0]
        sub = json.loads(path.read_text())
        companies.append({
            "ticker": ticker, "cik": sub["cik"], "name": sub["name"], "sic": sub.get("sic"),
            "sic_description": sub.get("sicDescription"), "fiscal_year_end": sub.get("fiscalYearEnd"),
            "state_of_incorporation": sub.get("stateOfIncorporation"), "entity_type": sub.get("entityType"),
            "category": sub.get("category"), "exchanges": sub.get("exchanges"), "tickers": sub.get("tickers"),
            "former_names": sub.get("formerNames"), "source_file": path.name,
        })
        recent = pd.DataFrame(sub["filings"]["recent"])
        recent.insert(0, "ticker", ticker)
        recent["source_file"] = path.name
        filings.append(recent)
    tables = {"raw_sec_facts": pd.DataFrame(facts), "raw_sec_concept": pd.DataFrame(concepts)}
    if companies:
        tables["raw_sec_company"] = pd.DataFrame(companies)
        tables["raw_sec_filing"] = pd.concat(filings, ignore_index=True)
    return tables


def read_fdic(batch_dir):
    tables = {}
    cohort = batch_dir / "institution_cohort.json"
    if cohort.exists():
        banks = pd.DataFrame(json.loads(cohort.read_text()))
        banks["source_file"] = cohort.name
        tables["raw_fdic_institution"] = banks
    rows = []
    for page in sorted(batch_dir.glob("financials/*/page_*.json")):
        content = json.loads(page.read_text())
        for item in content["data"]:
            record = dict(item["data"])          # each record is wrapped in "data"
            record["source_file"] = page.relative_to(batch_dir).as_posix()
            rows.append(record)
    tables["raw_fdic_financials"] = pd.DataFrame(rows)
    return tables


def read_fred(batch_dir):
    frames = []
    for path in sorted((batch_dir / "fred").glob("*.csv")):
        s = pd.read_csv(path, dtype=str)         # keep values as published text
        series_id = s.columns[1]
        s = s.rename(columns={series_id: "value"})
        s.insert(0, "series_id", series_id)
        s["source_file"] = path.relative_to(batch_dir).as_posix()
        frames.append(s)
    catalog = json.loads((batch_dir / "series_catalog.json").read_text())
    series = pd.DataFrame([{"series_id": k, "description": v} for k, v in catalog["series"].items()])
    series["source_file"] = "series_catalog.json"
    return {"raw_fred_observation": pd.concat(frames, ignore_index=True), "raw_fred_series": series}


if team["source"] == "SEC":
    tables = read_sec(batch_dir)
else:
    tables = read_fdic(batch_dir)
    if team["source"] == "FDIC+FRED":
        tables.update(read_fred(batch_dir))

tables = {name: to_text_if_nested(df) for name, df in tables.items()}
for name, df in tables.items():
    print(f"{name:24s} {len(df):>8,} rows  {df.shape[1]:>4} columns")

## 3. Save the raw tables to SQLite

Each table is appended to `team.db` with a `batch` column. The table `etl_batch` records every file that has been loaded, with its checksum and time, so running this cell again skips a batch that is already in the database instead of duplicating it.

In [ ]:
con = sqlite3.connect(DB_PATH)
con.execute("""
CREATE TABLE IF NOT EXISTS etl_batch (
    batch      TEXT NOT NULL,
    file_path  TEXT NOT NULL,
    sha256     TEXT NOT NULL,
    loaded_at  TEXT NOT NULL,
    PRIMARY KEY (batch, file_path)
)""")

already = set(pd.read_sql("SELECT file_path FROM etl_batch WHERE batch = ?", con, params=(BATCH,))["file_path"])
new_files = {f["path"]: f["sha256"] for f in manifest["files"] if f["path"] not in already}

if not new_files:
    print(f"Batch {BATCH} is already in {DB_PATH}: nothing to load")
else:
    loaded_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    with con:   # one transaction: everything is saved together, or nothing is
        for name, df in tables.items():
            rows = df[df["source_file"].isin(new_files)].assign(batch=BATCH)
            rows.to_sql(name, con, if_exists="append", index=False)
        con.executemany("INSERT INTO etl_batch VALUES (?, ?, ?, ?)",
                        [(BATCH, path, sha, loaded_at) for path, sha in new_files.items()])
    print(f"Loaded batch {BATCH}: {len(new_files)} files recorded in etl_batch")

# Every row in the raw tables must come from a file recorded in etl_batch
for name in tables:
    counts = pd.read_sql(f"SELECT batch, COUNT(*) AS n_rows FROM {name} GROUP BY batch", con)
    print(name, dict(zip(counts["batch"], counts["n_rows"])))

## 4. First look with SQL

The queries below only describe what you have. Everything you build next (normalized tables, star schema, KPIs, your own analysis) happens in your team's and your own notebooks, as your team instructions describe.

In [ ]:
print(pd.read_sql("SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name", con).to_string(index=False))

In [ ]:
if team["source"] == "SEC":
    query = """
        SELECT ticker,
               COUNT(DISTINCT concept) AS concepts,
               COUNT(*)                AS observations,
               MIN(filed)              AS first_filed,
               MAX(filed)              AS last_filed
        FROM raw_sec_facts
        GROUP BY ticker
        ORDER BY ticker
    """
else:
    query = """
        SELECT CERT,
               MIN(NAME)   AS name_in_financials,
               COUNT(*)    AS quarters,
               MIN(REPDTE) AS first_quarter,
               MAX(REPDTE) AS last_quarter
        FROM raw_fdic_financials
        GROUP BY CERT
        ORDER BY CERT
    """
pd.read_sql(query, con)

In [ ]:
if team["source"] == "FDIC+FRED":
    display(pd.read_sql("""
        SELECT series_id,
               COUNT(*)                            AS observations,
               SUM(value IS NULL OR value = '')    AS blank_values,
               MIN(observation_date)               AS first_date,
               MAX(observation_date)               AS last_date
        FROM raw_fred_observation
        GROUP BY series_id
        ORDER BY series_id
    """, con))
else:
    print("FRED data is only for Team 7.")

In [ ]:
con.close()
print("Done. Your raw data is in", DB_PATH.resolve())

## What next

1. Open your **team instructions document** and follow Week 1.
2. Create your team notebooks next to this one (`01_store.ipynb`, `02_model.ipynb`, `03_measure.ipynb`, `04_deliver.ipynb`) and your own notebook in `members/`.
3. Every notebook you write starts from `team.db` and must run top to bottom with **Restart & Run All**.
4. On November 30, set `BATCH = "B"` here and run this notebook again.